# NBA Playoff predictor
- We will train our model based off of 20 years of season data
- Features we will select include otrg, dtrg, pace, rest_days
- TRAIN_SEASONS = 2005-06 to 2021-22 (Large Sample good for model training)
- TEST_SEASONS  = 2022-23 to 2023-24 (Recent seasons would be better for not generalising)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from nba_api.stats.endpoints import leaguegamefinder, leaguestandings, leaguedashteamstats
import time

SEASONS = [f'{y}-{str(y+1)[-2:]}' for y in range(2003, 2024)]
DELAY = 0.6
OUT_DIR = Path('data')
OUT_DIR.mkdir(exist_ok=True)

HEADERS = {
    'Host':                  'stats.nba.com',
    'User-Agent':            'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Accept':                'application/json, text/plain, */*',
    'Accept-Language':       'en-US,en;q=0.9',
    'Referer':               'https://www.nba.com/',
    'x-nba-stats-origin':   'stats',
    'x-nba-stats-token':    'true',
    'Connection':            'keep-alive',

}

def scrape_playoff_games(seasons):
    all_games = []

    for season in seasons:
        try:
            resp = leaguegamefinder.LeagueGameFinder(
                season_nullable= season,
                season_type_nullable='Playoffs',
                league_id_nullable='00',
                player_or_team_abbreviation='T',
                headers=HEADERS,
                timeout=30   
            ).get_data_frames()[0]
        
            if resp.empty:
                print(f'[WARN]: No data for {season}')
                continue

            resp['SEASON'] = season
            all_games.append(
                resp[['SEASON', 'TEAM_ID', 'TEAM_ABBREVIATION',
                      'GAME_ID','GAME_DATE','MATCHUP','WL'
                      ]]
            )
        except Exception as e:
            print(f'[ERROR] {season}: {e}')
        
        time.sleep(DELAY)

    df = pd.concat(all_games,ignore_index=True)
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    df = df.sort_values(['SEASON', 'GAME_DATE', 'GAME_ID']).reset_index(drop=True)

    path = OUT_DIR / 'playoff_games.csv'
    df.to_csv(path, index=False)
    df.head(10)
    return df



In [ ]:
def scrape_seedings(seasons):
    all_standings = []

    for season in seasons:
        try:
            resp = leaguestandings.LeagueStandings(
                season = season,
                season_type='Regular Season',
                league_id='00',
                headers=HEADERS,
                timeout=30
            ).get_data_frames()[0]
            SEED_CANDIDATES = ['PlayoffSeeding', 'PlayoffRank', 'PLAYOFF_SEED']
            seed_col = next(
                (c for c in SEED_CANDIDATES if c in resp.columns),
                None
            )
            if seed_col is None:
                print(f'[WARN] no seed column found for {season}, cols: {resp.columns.tolist()}')
                continue

            resp = resp[['TeamID', seed_col]].copy()
            resp.columns = ['TEAM_ID', 'PLAYOFF_SEED']
            resp['SEASON'] = season
            all_standings.append(resp)
            
        except Exception as e:
            print(f'[ERROR] {season}: {e}')

        time.sleep(DELAY)

    df = pd.concat(all_standings, ignore_index=True)
    df['PLAYOFF_SEED'] = pd.to_numeric(df['PLAYOFF_SEED'], errors='coerce')
    df.head(10)
    return df

In [ ]:
def reconstruct_series(games, seedings):
    games = games.copy()
    
    # 1. Parse out the round number explicitly from the GAME_ID string
    # NBA playoff GAME_ID looks like: '0042300151' -> '1' represents Round 1
    games['GAME_ID_STR'] = games['GAME_ID'].astype(str)
    games['PLAYOFF_ROUND'] = games['GAME_ID_STR'].str[7:8].astype(int)
    
    # 2. Update SERIES_KEY to uniquely bundle specific rounds and matchups together
    games['SERIES_KEY'] = games['GAME_ID_STR'].str[:9]

    series_rows = []

    for (season, series_key), grp in games.groupby(['SEASON', 'SERIES_KEY']):
        # FILTER: 2 = Conference Semifinals, 3 = Conference Finals, 4 = NBA Finals
        playoff_round = grp['PLAYOFF_ROUND'].iloc[0]
        if playoff_round not in [2, 3, 4]:
            continue  # Skips the First Round (1) if you only want Semis onwards
            
        team_in_series = grp['TEAM_ID'].unique()
        if len(team_in_series) != 2:
            continue
            
        team_a_id, team_b_id = team_in_series

        wins_a = len(grp[(grp['TEAM_ID'] == team_a_id) & (grp['WL'] == 'W')])
        wins_b = len(grp[(grp['TEAM_ID'] == team_b_id) & (grp['WL'] == 'W')])

        # Ensure the series is actually finished (a team won 4 games)
        if wins_a != 4 and wins_b != 4:
            continue

        winner_id = team_a_id if wins_a == 4 else team_b_id
        games_played = wins_a + wins_b

        seed_lookup = (
            seedings[seedings['SEASON'] == season]
            .set_index('TEAM_ID')['PLAYOFF_SEED']
            .to_dict()
        )
        seed_a = seed_lookup.get(team_a_id)
        seed_b = seed_lookup.get(team_b_id)

        # Ensure Team A is ALWAYS the lower-numbered seed (The Favored/Higher Seed)
        if seed_a is not None and seed_b is not None and seed_a > seed_b:
            team_a_id, team_b_id = team_b_id, team_a_id
            seed_a, seed_b = seed_b, seed_a

        higher_seed_id = team_a_id
        higher_seed_wins = int(winner_id == higher_seed_id)
        abbrev = grp.set_index('TEAM_ID')['TEAM_ABBREVIATION'].to_dict()
        series_start = grp['GAME_DATE'].min()

        series_rows.append({
            'SEASON': season,
            'SERIES_KEY': series_key,
            'PLAYOFF_ROUND': playoff_round,  # Track the round name if needed
            'SERIES_START': series_start,
            'TEAM_A_ID': team_a_id,
            'TEAM_B_ID': team_b_id,
            'TEAM_A_ABB': abbrev.get(team_a_id),
            'TEAM_B_ABB': abbrev.get(team_b_id),
            'SEED_A': seed_a,
            'SEED_B': seed_b,
            'WINS_A': wins_a if winner_id == team_a_id else wins_b,
            'WINS_B': wins_b if winner_id == team_a_id else wins_a,
            'GAMES_PLAYED': games_played,
            'WINNER_ID': winner_id,
            'HIGHER_SEED_WINS': higher_seed_wins
        })

    df = pd.DataFrame(series_rows).sort_values(['SEASON', 'SERIES_START'])
    path = OUT_DIR / 'playoff_series.csv'
    df.to_csv(path, index=False)
    print(f"Successfully processed {len(df)} playoff series rows.")
    return df

In [ ]:
def scrape_reg_seasons_stats(seasons):
    all_stats = []

    for season in seasons:
        try:
            df = leaguedashteamstats.LeagueDashTeamStats(
                season = season,
                season_type_all_star='Regular Season',
                measure_type_detailed_defense='Advanced',
                per_mode_detailed='PerGame',
                headers=HEADERS,
                timeout=30
            ).get_data_frames()[0]

            col_map = {}
            for c in df.columns:
                cu = c.upper()
                if 'OFF_RATING' in cu and 'ORTG' not in col_map.values():
                    col_map[c] = 'ORTG'
                elif 'DEF_RATING' in cu and 'DRTG' not in col_map.values():
                    col_map[c] = 'DRTG'
                elif cu in ('PACE', 'E_PACE') and 'PACE' not in col_map.values():
                    col_map[c] = 'PACE'
                elif 'NET_RATING' in cu and 'NET_RTG' not in col_map.values():
                    col_map[c] = 'NET_RTG'
                
            df = df.rename(columns=col_map)

            required = {'ORTG','DRTG','PACE'}
            missing = required - set(df.columns)
            if missing:
                print(f'[WARN]: missing columns {missing} for {season} — skipping')
                print(f'Available: {df.columns.tolist()}')

                continue
            df['SEASON'] = season
            keep = ['SEASON', 'TEAM_ID', 'TEAM_NAME', 'W', 'L', 'W_PCT',
                    'ORTG', 'DRTG', 'PACE', 'NET_RTG']
            keep = [c for c in keep if c in df.columns]
            all_stats.append(df[keep])

        except Exception as e:
            print(f'[ERROR] {season}: {e}')
        
        time.sleep(DELAY)

    stats = pd.concat(all_stats,ignore_index=True)

    stats = stats.drop_duplicates(subset=["SEASON", "TEAM_ID"], keep="first")

    path = OUT_DIR / 'reg_season_stats.csv'
    stats.to_csv(path, index=False)
    return stats       

In [ ]:
def build_series_features(series_path, stats):
    series    = pd.read_csv(series_path, parse_dates=["SERIES_START"])
    stats_idx = stats.set_index(["SEASON", "TEAM_ID"])
    rows      = []
    skipped   = 0

    for _, s in series.iterrows():
        season = s["SEASON"]
        a_id   = s["TEAM_A_ID"]
        b_id   = s["TEAM_B_ID"]

        a = stats_idx.loc[(season, a_id)]
        b = stats_idx.loc[(season, b_id)]

        has_net = "NET_RTG" in a.index

        rows.append({
            "SEASON":           season,
            "SERIES_KEY":       s["SERIES_KEY"],
            "SERIES_START":     s["SERIES_START"],
            "TEAM_A_ABB":       s["TEAM_A_ABB"],
            "TEAM_B_ABB":       s["TEAM_B_ABB"],
            "SEED_A":           s["SEED_A"],
            "SEED_B":           s["SEED_B"],
            "ORTG_A":           a["ORTG"],
            "DRTG_A":           a["DRTG"],
            "PACE_A":           a["PACE"],
            "W_PCT_A":          a["W_PCT"],
            "ORTG_B":           b["ORTG"],
            "DRTG_B":           b["DRTG"],
            "PACE_B":           b["PACE"],
            "W_PCT_B":          b["W_PCT"],
            "ORTG_DIFF":        a["ORTG"]  - b["ORTG"],
            "DRTG_DIFF":        a["DRTG"]  - b["DRTG"],
            "PACE_DIFF":        a["PACE"]  - b["PACE"],
            "NET_RTG_DIFF":     a["NET_RTG"] - b["NET_RTG"] if has_net else np.nan,
            "W_PCT_DIFF":       a["W_PCT"] - b["W_PCT"],
            "SEED_DIFF":        s["SEED_A"] - s["SEED_B"],
            "HIGHER_SEED_WINS": s["HIGHER_SEED_WINS"],
        })

    features = pd.DataFrame(rows).sort_values(["SEASON", "SERIES_START"])
    features.to_csv(OUT_DIR / "series_features.csv", index=False)

    return features


In [ ]:
def validate_features(df: pd.DataFrame) -> None:
    feature_cols = ['ORTG_DIFF', 'DRTG_DIFF', 'PACE_DIFF', 'NET_RTG_DIFF',
                    'W_PCT_DIFF', 'SEED_DIFF']
 
    print(f'\n  Shape:  {df.shape}')
    print(f'  Seasons: {df['SEASON'].nunique()}  ({df['SEASON'].min()} → {df['SEASON'].max()})')
 
    print(f'\n  Target distribution:')
    vc = df['HIGHER_SEED_WINS'].value_counts(normalize=True)
    print(f'    Higher seed wins: {vc.get(1, 0):.1%}')
    print(f'    Upset:            {vc.get(0, 0):.1%}')
 
    print(f'\n  Nulls per feature:')
    print(df[feature_cols].isnull().sum().to_string())
 
    print(f'\n  Feature summary stats:')
    print(df[feature_cols].describe().round(2).to_string())
 
    bad_seeds = (df['SEED_DIFF'] > 0).sum()
    if bad_seeds:
        print(f'\n  [WARN]: {bad_seeds} rows where SEED_DIFF > 0 (team orientation wrong)')
    else:
        print(f'\n  SEED_DIFF always <= 0 (team orientation correct)')



In [ ]:


games = scrape_playoff_games(SEASONS)
seedings = scrape_seedings(SEASONS)
series = reconstruct_series(games,seedings)
scrape_reg_seasons_stats(SEASONS)

stats = pd.read_csv('./data/reg_season_stats.csv')
build_series_features(OUT_DIR / 'playoff_series.csv', stats)
# features = build_series_features(OUT_DIR / 'playoff_series.csv', stats)
# validate_features(features)